In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Users\polok\deep-learning-assignment-1")
DATA_DIR = PROJECT_ROOT / "data"

print("Project exists:", PROJECT_ROOT.exists())
print("Data exists:", DATA_DIR.exists())
print("Train exists:", (DATA_DIR / "train_data.csv").exists())

train = pd.read_csv(DATA_DIR / "train_data.csv")
test = pd.read_csv(DATA_DIR / "test_data.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Project exists: True
Data exists: True
Train exists: True
Train: (5531451, 190)
Test: (11363762, 190)


**GENERAL ANALYSIS**

Missing values: how many do we have?

In [3]:
missing_summary = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_percent": train.isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

print("Total missing values:")
print(train.isna().sum().sum())

print("\nFeatures with missing values:")
print((missing_summary["missing_count"] > 0).sum())

print("\nFeatures without missing values:")
print((missing_summary["missing_count"] == 0).sum())

display(missing_summary.head(30))

Total missing values:
160858968

Features with missing values:
122

Features without missing values:
68


,missing_count,missing_percent
D_87,5527586,99.930127
D_88,5525447,99.891457
D_108,5502513,99.476846
D_110,5500117,99.433530
D_111,5500117,99.433530
B_39,5497819,99.391986
D_73,5475595,98.990211
B_42,5459973,98.707789
D_135,5336752,96.480146
D_134,5336752,96.480146


In [4]:
missing_summary["missing_group"] = pd.cut(
    missing_summary["missing_percent"],
    bins=[-1, 0, 10, 30, 50, 80, 100],
    labels=[
        "No missing",
        "0-10%",
        "10-30%",
        "30-50%",
        "50-80%",
        "80-100%"
    ]
)

display(
    missing_summary["missing_group"]
    .value_counts()
    .sort_index()
)

missing_group
No missing    68
0-10%         83
10-30%         8
30-50%         1
50-80%         7
80-100%       23
Name: count, dtype: int64

What type is each feature?

In [5]:
categorical_cols = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68"
]

def get_semantic_type(column):
    if column == "customer_ID":
        return "identifier"

    if column == "S_2":
        return "date"

    if column in categorical_cols:
        return "categorical"

    return "numerical"


feature_types = pd.DataFrame({
    "feature": train.columns,
    "pandas_dtype": train.dtypes.astype(str).values,
    "semantic_type": [
        get_semantic_type(col)
        for col in train.columns
    ],
    "unique_values": [
        train[col].nunique(dropna=False)
        for col in train.columns
    ]
})

display(feature_types)

,feature,pandas_dtype,semantic_type,unique_values
0,customer_ID,str,identifier,458913
1,S_2,str,date,396
2,P_2,float64,numerical,5485467
3,D_39,float64,numerical,5531451
4,B_1,float64,numerical,5531451
...,...,...,...,...
185,D_141,float64,numerical,5429904
186,D_142,float64,numerical,944409
187,D_143,float64,numerical,5429904
188,D_144,float64,numerical,5490725


Are there duplicate rows?

In [25]:
print("QUESTION: Are there duplicate customer-statement records?")

duplicate_rows = train.duplicated().sum()

duplicate_customer_dates = train.duplicated(
    subset=["customer_ID", "S_2"]
).sum()

print("Exact duplicate rows:", duplicate_rows)
print(
    "Duplicate customer/date combinations:",
    duplicate_customer_dates
)

QUESTION: Are there duplicate customer-statement records?
Exact duplicate rows: 0
Duplicate customer/date combinations: 0


Are categorical feature values associated with different default rates?

In [26]:
print(
    "QUESTION: Are categorical feature values "
    "associated with different default rates?"
)

latest_idx = (
    train
    .groupby("customer_ID")["S_2"]
    .idxmax()
)

latest = (
    train.loc[latest_idx]
    .merge(labels, on="customer_ID", how="left")
)

for col in categorical_cols:

    result = (
        latest
        .groupby(col, dropna=False)
        .agg(
            customers=("customer_ID", "count"),
            default_rate=("target", "mean")
        )
    )

    result["default_rate_percent"] = (
        result["default_rate"] * 100
    )

    print(f"\n===== {col} =====")
    display(
        result.sort_values(
            "default_rate",
            ascending=False
        )
    )

QUESTION: Are categorical feature values associated with different default rates?

===== B_30 =====


,customers,default_rate,default_rate_percent
B_30,,,
1.0,71127,0.662421,66.242074
2.0,4668,0.610540,61.053985
NaN,31,0.225806,22.580645
0.0,383087,0.179737,17.973724



===== B_38 =====


,customers,default_rate,default_rate_percent
B_38,,,
4.0,25901,0.720358,72.035829
6.0,15862,0.675073,67.507250
5.0,38581,0.581944,58.194448
7.0,26469,0.549473,54.947297
3.0,104242,0.345427,34.542699
NaN,31,0.225806,22.580645
1.0,95272,0.090226,9.022588
2.0,152555,0.051490,5.148963



===== D_114 =====


...

,customers,default_rate,default_rate_percent
D_64,,,
U,132722,0.363903,36.390350
R,71532,0.334396,33.439579
NaN,10037,0.295208,29.520773
O,244622,0.178426,17.842631



===== D_66 =====


,customers,default_rate,default_rate_percent
D_66,,,
NaN,406331,0.267319,26.731901
1.0,52582,0.194135,19.413488



===== D_68 =====


,customers,default_rate,default_rate_percent
D_68,,,
1.0,10223,0.427076,42.707620
2.0,19318,0.412620,41.262035
3.0,41898,0.391761,39.176094
4.0,41410,0.365830,36.582951
NaN,9012,0.301376,30.137594
5.0,105517,0.281253,28.125326
6.0,231535,0.183709,18.370873


**TARGET ANALYSIS**

Explore the target imbalance in train.

In [12]:
labels = pd.read_csv(DATA_DIR / "train_labels.csv")

result = pd.DataFrame({
    "customers": labels["target"].value_counts().sort_index(),
    "percentage": (
        labels["target"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )
})

print("QUESTION: Is the target imbalanced?")
display(result)

QUESTION: Is the target imbalanced?


,customers,percentage
target,,
0,340085,74.106639
1,118828,25.893361


Do customers with fewer entries default more often?

In [13]:
statement_counts = (
    train
    .groupby("customer_ID")
    .size()
    .rename("n_statements")
    .reset_index()
)

data = statement_counts.merge(
    labels,
    on="customer_ID",
    how="left"
)

result = (
    data
    .groupby("n_statements")
    .agg(
        customers=("customer_ID", "count"),
        default_rate=("target", "mean")
    )
)

result["default_rate_percent"] = (
    result["default_rate"] * 100
)

print("QUESTION: Do customers with fewer statements default more often?")
display(result)


QUESTION: Do customers with fewer statements default more often?


,customers,default_rate,default_rate_percent
n_statements,,,
1,5120,0.335742,33.574219
2,6098,0.318465,31.846507
3,5778,0.358602,35.860159
4,4673,0.416221,41.622084
5,4671,0.392635,39.263541
6,5515,0.387670,38.766999
7,5198,0.418430,41.843017
8,6110,0.447300,44.729951
9,6411,0.450164,45.016378


Is missinng values assosiated with default?

In [14]:
latest_idx = (
    train
    .groupby("customer_ID")["S_2"]
    .idxmax()
)

latest = (
    train.loc[latest_idx]
    .merge(labels, on="customer_ID", how="left")
)

results = []

for col in train.columns:
    if col in ["customer_ID", "S_2"]:
        continue

    is_missing = latest[col].isna()

    if is_missing.any() and (~is_missing).any():

        results.append({
            "feature": col,

            "missing_percent":
                is_missing.mean() * 100,

            "default_rate_missing":
                latest.loc[
                    is_missing,
                    "target"
                ].mean(),

            "default_rate_present":
                latest.loc[
                    ~is_missing,
                    "target"
                ].mean()
        })

result = pd.DataFrame(results)

result["difference"] = (
    result["default_rate_missing"]
    - result["default_rate_present"]
)

result = result.sort_values(
    "difference",
    key=abs,
    ascending=False
)

print("QUESTION: Is feature missingness associated with default?")
display(result.head(30))

QUESTION: Is feature missingness associated with default?


,feature,missing_percent,default_rate_missing,default_rate_present,difference
56,D_87,99.859450,0.257978,0.937984,-0.680007
57,D_88,99.819792,0.257797,0.888755,-0.630958
64,S_25,0.309645,0.751583,0.257403,0.494180
62,S_23,0.009588,0.659091,0.258895,0.400196
47,R_9,94.126773,0.236996,0.610507,-0.373511
70,R_26,88.855622,0.217527,0.589074,-0.371547
102,D_137,96.427427,0.246209,0.602379,-0.356170
101,D_136,96.427427,0.246209,0.602379,-0.356170
103,D_138,96.427427,0.246209,0.602379,-0.356170
100,D_135,96.427427,0.246209,0.602379,-0.356170


Is recent behavior more informative than long-term behavior?

In [17]:
from sklearn.metrics import roc_auc_score

# Mean over entire available history
full_mean = (
    train
    .groupby("customer_ID")[FEATURE]
    .mean()
    .rename("full_history_mean")
)

# Mean of only latest 3 statements
recent3 = (
    train
    .groupby("customer_ID")
    .tail(3)
    .groupby("customer_ID")[FEATURE]
    .mean()
    .rename("recent3_mean")
)

data = pd.concat(
    [full_mean, recent3],
    axis=1
).reset_index()

data = data.merge(
    labels,
    on="customer_ID",
    how="left"
)

def discrimination_auc(feature):
    valid = data[feature].notna()

    auc = roc_auc_score(
        data.loc[valid, "target"],
        data.loc[valid, feature]
    )

    return max(auc, 1 - auc)

full_auc = discrimination_auc(
    "full_history_mean"
)

recent_auc = discrimination_auc(
    "recent3_mean"
)

print("QUESTION: Is recent behavior more predictive than long-term behavior?")
print()
print("Feature:", FEATURE)
print("Full-history mean AUC:", round(full_auc, 4))
print("Recent-3 mean AUC:", round(recent_auc, 4))
print("Difference:", round(recent_auc - full_auc, 4))

QUESTION: Is recent behavior more predictive than long-term behavior?

Feature: P_2
Full-history mean AUC: 0.8995
Recent-3 mean AUC: 0.9134
Difference: 0.0139


Meaning: average of the latest 3 statements separates defaulting and non-defaulting customers a little better than the average across the whole customer history.

This uses ROC-AUC as a measure of univariate discrimination [how informative one feature is by itself]:

In [18]:
latest_idx = (
    train
    .groupby("customer_ID")["S_2"]
    .idxmax()
)

latest = (
    train.loc[latest_idx]
    .merge(labels, on="customer_ID", how="left")
)

numeric_cols = [
    col
    for col in latest.select_dtypes(include=np.number).columns
    if col not in categorical_cols + ["target"]
]

results = []

for col in numeric_cols:

    valid = latest[col].notna()

    if latest.loc[valid, col].nunique() < 2:
        continue

    auc = roc_auc_score(
        latest.loc[valid, "target"],
        latest.loc[valid, col]
    )

    discrimination = max(
        auc,
        1 - auc
    )

    results.append({
        "feature": col,
        "auc": auc,
        "discrimination": discrimination,
        "available_percent": valid.mean() * 100
    })

result = (
    pd.DataFrame(results)
    .sort_values(
        "discrimination",
        ascending=False
    )
)

print("QUESTION: Which numerical features individually separate defaults best?")
display(result.head(30))

QUESTION: Which numerical features individually separate defaults best?


,feature,auc,discrimination,available_percent
0,P_2,0.083183,0.916817,99.353036
17,D_48,0.877511,0.877511,87.363182
47,D_61,0.850645,0.850645,89.464670
54,B_18,0.150650,0.849350,100.000000
75,D_77,0.150715,0.849285,53.403586
20,B_7,0.846678,0.846678,100.000000
158,B_42,0.153636,0.846364,1.338380
71,D_75,0.845796,0.845796,100.000000
67,B_23,0.844992,0.844992,100.000000
10,D_44,0.844349,0.844349,95.141781


Are the strongest numerical features correlated and therefore redundant?

In [27]:
print(
    "QUESTION: Are the strongest numerical features "
    "highly correlated with each other?"
)

top_features = [
    "P_2",
    "D_48",
    "D_61",
    "B_18",
    "D_77",
    "B_7",
    "D_75",
    "B_23",
    "D_44",
    "D_55",
    "B_2",
    "B_3",
    "B_6",
    "B_1"
]

corr = latest[top_features].corr(
    method="spearman"
)

pairs = []

for i in range(len(top_features)):
    for j in range(i + 1, len(top_features)):

        value = corr.iloc[i, j]

        pairs.append({
            "feature_1": top_features[i],
            "feature_2": top_features[j],
            "correlation": value,
            "abs_correlation": abs(value)
        })

correlation_pairs = (
    pd.DataFrame(pairs)
    .sort_values(
        "abs_correlation",
        ascending=False
    )
)

display(correlation_pairs.head(30))

QUESTION: Are the strongest numerical features highly correlated with each other?


,feature_1,feature_2,correlation,abs_correlation
56,B_7,B_23,0.991508,0.991508
20,D_48,D_55,0.900122,0.900122
13,D_48,D_61,0.878366,0.878366
38,B_18,D_75,-0.861410,0.861410
31,D_61,D_55,0.859316,0.859316
61,B_7,B_6,-0.858958,0.858958
74,B_23,B_6,-0.857237,0.857237
39,B_18,B_23,-0.822085,0.822085
63,D_75,B_23,0.813971,0.813971
37,B_18,B_7,-0.813166,0.813166


Are important numerical features strongly skewed or affected by extreme values?

In [28]:
print(
    "QUESTION: Are important numerical features "
    "strongly skewed or affected by extreme values?"
)

features = [
    "P_2",
    "D_48",
    "D_61",
    "B_18",
    "B_7",
    "D_75",
    "B_23",
    "D_44",
    "B_2",
    "B_3",
    "B_1"
]

distribution_summary = []

for col in features:

    x = latest[col].dropna()

    distribution_summary.append({
        "feature": col,
        "min": x.min(),
        "p01": x.quantile(0.01),
        "p25": x.quantile(0.25),
        "median": x.median(),
        "p75": x.quantile(0.75),
        "p99": x.quantile(0.99),
        "max": x.max(),
        "skew": x.skew()
    })

distribution_summary = pd.DataFrame(
    distribution_summary
)

display(distribution_summary)

QUESTION: Are important numerical features strongly skewed or affected by extreme values?


,feature,min,p01,p25,median,p75,p99,max,skew
0,P_2,-4.589548e-01,-0.076811,0.449332,0.682559,0.862186,1.005943,1.009998,-0.681382
1,D_48,-9.603428e-03,0.001316,0.084098,0.317044,0.734500,1.043498,8.964578,0.491508
2,D_61,-9.256992e-03,0.000736,0.099558,0.414736,0.814900,1.034797,46.417835,5.945572
3,B_18,2.327584e-07,0.002915,0.200543,0.592520,1.003003,1.009717,1.010000,-0.059852
4,B_7,-2.652748e+00,0.002531,0.028956,0.084739,0.306708,1.035362,1.252748,1.647253
5,D_75,1.355373e-07,0.000281,0.006982,0.075507,0.272342,1.069385,4.076651,2.202095
6,B_23,5.498423e-07,0.001434,0.018298,0.068034,0.283097,1.048791,1.544252,1.731754
7,D_44,1.031370e-08,0.000162,0.003996,0.008030,0.134153,1.128850,4.005466,2.823011
8,B_2,2.024839e-07,0.001735,0.070912,0.813405,1.001955,1.009674,1.010000,-0.420541
9,B_3,7.574315e-08,0.000266,0.005484,0.010830,0.210661,1.041620,1.596532,1.945054


Do train and test cover different time periods?

In [23]:
print("QUESTION: Do train and test cover different time periods?")
print()

train_dates = pd.to_datetime(train["S_2"])
test_dates = pd.to_datetime(test["S_2"])

print("Train:")
print(train_dates.min(), "->", train_dates.max())

print("\nTest:")
print(test_dates.min(), "->", test_dates.max())

QUESTION: Do train and test cover different time periods?

Train:
2017-03-01 00:00:00 -> 2018-03-31 00:00:00

Test:
2018-04-01 00:00:00 -> 2019-10-31 00:00:00


Can we safely process the dataset into one row per customer?

In [ ]:
print(
    "QUESTION: Can customer history be compressed into one row "
    "without losing important predictive information?"
)

features = [
    "P_2",
    "D_48",
    "D_61",
    "B_18",
    "B_7",
    "D_75",
    "B_23",
    "D_44",
    "B_2",
    "B_3",
    "B_1"
]

train["S_2"] = pd.to_datetime(train["S_2"])

train_sorted = train.sort_values(
    ["customer_ID", "S_2"]
)

results = []

for feature in features:

    grouped = train_sorted.groupby("customer_ID")[feature]

    customer_features = pd.DataFrame({
        "last": grouped.last(),
        "mean": grouped.mean(),
        "std": grouped.std(),
        "first": grouped.first()
    })

    customer_features["change"] = (
        customer_features["last"]
        - customer_features["first"]
    )

    recent3_mean = (
        train_sorted
        .groupby("customer_ID")
        .tail(3)
        .groupby("customer_ID")[feature]
        .mean()
    )

    customer_features["recent3_mean"] = recent3_mean

    customer_features = (
        customer_features
        .reset_index()
        .merge(
            labels,
            on="customer_ID",
            how="left"
        )
    )

    for representation in [
        "last",
        "mean",
        "std",
        "change",
        "recent3_mean"
    ]:

        valid = (
            customer_features[representation].notna()
            & customer_features["target"].notna()
        )

        if customer_features.loc[
            valid, representation
        ].nunique() < 2:
            continue

        auc = roc_auc_score(
            customer_features.loc[valid, "target"],
            customer_features.loc[valid, representation]
        )

        discrimination = max(
            auc,
            1 - auc
        )

        results.append({
            "feature": feature,
            "representation": representation,
            "discrimination": discrimination
        })

result = pd.DataFrame(results)

comparison = result.pivot(
    index="feature",
    columns="representation",
    values="discrimination"
)

display(comparison.round(4))

QUESTION: Can customer history be compressed into one row without losing important predictive information?


representation,change,last,mean,recent3_mean,std
feature,,,,,
B_1,0.6893,0.8354,0.8182,0.8337,0.7153
B_18,0.7008,0.8494,0.8283,0.8488,0.5603
B_2,0.6653,0.8394,0.8368,0.8533,0.6482
B_23,0.6801,0.8450,0.8234,0.8411,0.7806
B_3,0.7374,0.8391,0.8235,0.8375,0.8292
B_7,0.6705,0.8467,0.8264,0.8431,0.7731
D_44,0.6782,0.8431,0.8524,0.8523,0.7965
D_48,0.6788,0.8793,0.8637,0.8770,0.6125
D_61,0.6347,0.8535,0.8397,0.8523,0.5622


Meaning: Customer history can be reasonably compressed into one row per customer for the ML baseline if several summary features are kept rather than only the latest value.

**TRAIN vs TEST FOR RELIABLE VALIDATION**

Do train and test have different missing-value patterns?

In [29]:
print(
    "QUESTION: Do train and test have different "
    "missing-value patterns?"
)

train_missing = (
    train
    .drop(columns=["customer_ID", "S_2"])
    .isna()
    .mean()
)

test_missing = (
    test
    .drop(columns=["customer_ID", "S_2"])
    .isna()
    .mean()
)

missing_drift = pd.DataFrame({
    "train_missing": train_missing,
    "test_missing": test_missing
})

missing_drift["difference"] = (
    missing_drift["test_missing"]
    - missing_drift["train_missing"]
)

missing_drift["abs_difference"] = (
    missing_drift["difference"].abs()
)

display(
    missing_drift
    .sort_values(
        "abs_difference",
        ascending=False
    )
    .head(30)
)

QUESTION: Do train and test have different missing-value patterns?


,train_missing,test_missing,difference,abs_difference
B_29,0.931046,0.750382,-0.180664,0.180664
S_9,0.530357,0.350709,-0.179648,0.179648
D_56,0.540716,0.502713,-0.038003,0.038003
D_77,0.454476,0.490403,0.035927,0.035927
R_27,0.023267,0.058388,0.035120,0.035120
S_27,0.253267,0.220022,-0.033245,0.033245
D_55,0.033409,0.065381,0.031971,0.031971
S_3,0.184498,0.153203,-0.031295,0.031295
S_7,0.184498,0.153203,-0.031295,0.031295
R_26,0.889847,0.860033,-0.029814,0.029814


Shifted feature distribution

In [30]:
train["S_2"] = pd.to_datetime(train["S_2"])
test["S_2"] = pd.to_datetime(test["S_2"])

train_latest_idx = (
    train
    .groupby("customer_ID")["S_2"]
    .idxmax()
)

test_latest_idx = (
    test
    .groupby("customer_ID")["S_2"]
    .idxmax()
)

train_latest = train.loc[train_latest_idx].copy()
test_latest = test.loc[test_latest_idx].copy()

print("Train latest:", train_latest.shape)
print("Test latest:", test_latest.shape)

Train latest: (458913, 190)
Test latest: (924621, 190)


In [31]:
from scipy.stats import ks_2samp

print("QUESTION: Which numerical features shifted most between train and test?")

categorical_cols = [
    "B_30", "B_38", "D_114", "D_116",
    "D_117", "D_120", "D_126",
    "D_63", "D_64", "D_66", "D_68"
]

numeric_cols = [
    c for c in train_latest.select_dtypes(include=np.number).columns
    if c not in categorical_cols
]

results = []

for col in numeric_cols:

    train_values = train_latest[col].dropna()
    test_values = test_latest[col].dropna()

    if len(train_values) == 0 or len(test_values) == 0:
        continue

    # Sample for speed
    train_sample = train_values.sample(
        n=min(100_000, len(train_values)),
        random_state=42
    )

    test_sample = test_values.sample(
        n=min(100_000, len(test_values)),
        random_state=42
    )

    ks = ks_2samp(
        train_sample,
        test_sample
    ).statistic

    results.append({
        "feature": col,
        "ks_statistic": ks,
        "train_median": train_values.median(),
        "test_median": test_values.median()
    })

numerical_drift = (
    pd.DataFrame(results)
    .sort_values("ks_statistic", ascending=False)
)

display(numerical_drift.head(30))

QUESTION: Which numerical features shifted most between train and test?


,feature,ks_statistic,train_median,test_median
49,S_11,0.435330,0.324229,0.440255
45,D_59,0.385320,0.484897,0.521315
142,B_39,0.269981,0.225312,0.392697
66,S_15,0.181780,0.403714,0.309684
104,D_88,0.124744,0.112503,0.075954
43,S_9,0.121990,0.011768,0.009265
8,D_42,0.121573,0.108293,0.065649
150,D_121,0.117020,0.594958,0.602402
125,S_24,0.101910,0.948790,0.949833
53,B_17,0.088610,0.903445,0.932141


Meaning: some feature distribution changed dramatically between train/test.